# 의료 논문 RAG 질의응답 (PubMedQA) - 전체 재현

의학 논문 초록을 근거로 임상 질문에 yes/no/maybe로 답하는 RAG 파이프라인.
**유료 API 키 없이** 무료 오픈모델로 Colab에서 전 과정 재현.

- 런타임 > 런타임 유형 변경 > **GPU(T4)** 선택 후 실행
- 전체 실행 시간: 약 10~15분
- 마지막 셀에서 결과를 zip으로 다운로드 → repo의 `results/`에 복사


In [ ]:
!pip -q install datasets sentence-transformers faiss-cpu anthropic transformers accelerate


## 1. 데이터 준비

PubMedQA(pqa_labeled, 전문가 라벨 1,000건)를 받아
검색 코퍼스(모든 초록 문단)와 질문셋(dev/test)으로 나눈다.
`src/prepare_data.py`와 동일한 로직.


In [ ]:
import json, random
from datasets import load_dataset

ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
print(ds)

corpus, questions, seen = [], [], set()
for i, row in enumerate(ds):
    pubid = str(row.get("pubid", i))
    ctx = row["context"]
    passages = ctx["contexts"] if isinstance(ctx, dict) else ctx
    labels = ctx.get("labels", []) if isinstance(ctx, dict) else []
    for j, text in enumerate(passages):
        text = (text or "").strip()
        if not text or (pubid, j) in seen:
            continue
        seen.add((pubid, j))
        corpus.append({"pid": f"{pubid}_{j}", "pubid": pubid,
                       "section": labels[j] if j < len(labels) else "",
                       "text": text})
    questions.append({"qid": pubid, "pubid": pubid,
                      "question": row["question"].strip(),
                      "decision": row["final_decision"].strip().lower(),
                      "long_answer": row.get("long_answer", "").strip()})

random.Random(42).shuffle(questions)
test = questions[:500]
print(f"코퍼스 문단 {len(corpus)} / 질문 {len(questions)} (test {len(test)})")

from collections import Counter
print("정답 분포:", Counter(q["decision"] for q in questions))


## 2. 검색기 (Retriever)

의료 도메인 문장 임베딩 모델로 코퍼스를 임베딩하고 FAISS로 검색.
임베딩을 정규화해 내적=코사인 유사도가 되게 함.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

corpus_texts = [c["text"] for c in corpus]
emb = embedder.encode(corpus_texts, batch_size=64, show_progress_bar=True,
                      convert_to_numpy=True, normalize_embeddings=True).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("index size:", index.ntotal)

def retrieve(question, k=5):
    qv = embedder.encode([question], normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    scores, idx = index.search(qv, k)
    return [{**corpus[int(i)], "score": float(s), "rank": r}
            for r, (i, s) in enumerate(zip(idx[0], scores[0]))]

retrieve("Is aspirin associated with reduced cardiovascular risk?", k=3)


## 3. 검색 성능 - recall@k

질문과 같은 논문(pubid)의 문단이 top-k 안에 들어오는 비율.


In [ ]:
def recall_at_k(qs, ks=(1,3,5,10)):
    kmax=max(ks); hit={k:0 for k in ks}
    for q in qs:
        got = retrieve(q["question"], k=kmax)
        ranks=[r["rank"] for r in got if r["pubid"]==q["pubid"]]
        first=min(ranks) if ranks else 10**9
        for k in ks:
            if first<k: hit[k]+=1
    return {k: hit[k]/len(qs) for k in ks}

recall = recall_at_k(test)
print("recall@k:", {k: round(v,4) for k,v in recall.items()})


## 4. 생성기 (Generator)

검색된 문단만 근거로 yes/no/maybe와 한 줄 이유를 생성.

생성기는 둘 중 하나를 고름:
- **Claude API (기본, 권장)** — 공고의 "LLM API"에 해당. 품질 좋고 GPU 불필요.
  아래 셀 실행 시 API 키를 입력받음 (화면/코드에 노출되지 않음).
- 오픈모델(Qwen2.5-1.5B) — 키 입력을 비우면 자동으로 이쪽 사용. GPU 필요.

**주의: API 키를 노트북/깃허브에 그대로 적어 저장하지 말 것.** 아래는
getpass로 입력받아 메모리에만 둠.


In [ ]:
import re, getpass, os

PROMPT = ("You are a biomedical question answering assistant.\n"
          "Use ONLY the provided research abstract excerpts to answer.\n"
          "Answer with exactly one of: yes, no, maybe. Then a one-sentence reason.\n\n"
          "Context:\n{context}\n\nQuestion: {question}\n\nAnswer (yes/no/maybe) and reason:")

def build_prompt(q, passages, max_chars=1800):
    ctx=""
    for p in passages:
        piece=f"- {p['text'].strip()}\n"
        if len(ctx)+len(piece)>max_chars: break
        ctx+=piece
    return PROMPT.format(context=ctx.strip(), question=q.strip())

def parse_decision(text):
    low=text.strip().lower()
    m=re.match(r"[^a-z]*\b(yes|no|maybe)\b", low)
    if m: return m.group(1)
    for d in ["yes","no","maybe"]:
        if re.search(rf"\b{d}\b", low): return d
    return "maybe"

# 키를 입력하면 Claude, 비우고 Enter면 오픈모델
_key = getpass.getpass("ANTHROPIC_API_KEY (없으면 그냥 Enter -> 오픈모델): ").strip()

if _key:
    from anthropic import Anthropic
    os.environ["ANTHROPIC_API_KEY"] = _key
    client = Anthropic()
    CLAUDE_MODEL = "claude-3-5-haiku-latest"   # 계정에 맞는 모델로 바꿔도 됨
    BACKEND = "claude"
    def generate(prompt):
        msg = client.messages.create(model=CLAUDE_MODEL, max_tokens=64,
                                     messages=[{"role":"user","content":prompt}])
        return "".join(b.text for b in msg.content if b.type=="text").strip()
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL="Qwen/Qwen2.5-1.5B-Instruct"
    tok=AutoTokenizer.from_pretrained(MODEL)
    lm=AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
    BACKEND="hf"
    def generate(prompt):
        msgs=[{"role":"user","content":prompt}]
        text=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp=tok(text, return_tensors="pt").to(lm.device)
        with torch.no_grad():
            out=lm.generate(**inp, max_new_tokens=64, do_sample=False,
                            pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("생성기 백엔드:", BACKEND)

def answer(question, k=5):
    ps=retrieve(question, k=k)
    raw=generate(build_prompt(question, ps))
    return {"decision":parse_decision(raw), "raw":raw, "retrieved":ps}

ex = answer(test[0]["question"])
print("Q:", test[0]["question"])
print("정답:", test[0]["decision"], "| 예측:", ex["decision"])
print("생성:", ex["raw"])


## 5. 최종 답변 정확도 + 저장

test 500건에 대해 RAG 파이프라인을 돌려 정확도와 confusion matrix 산출.
(500건 생성이라 몇 분 걸림)


In [ ]:
import os
os.makedirs("results_export", exist_ok=True)
labels=["yes","no","maybe"]
cm={a:{b:0 for b in labels} for a in labels}
correct=0; preds=[]
from tqdm import tqdm
for q in tqdm(test):
    res=answer(q["question"])
    pred, gold = res["decision"], q["decision"]
    preds.append({"qid":q["qid"],"gold":gold,"pred":pred,"raw":res["raw"]})
    if gold in labels and pred in labels: cm[gold][pred]+=1
    if pred==gold: correct+=1
acc=correct/len(test)
print(f"answer accuracy: {acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt, json

# recall@k
ks=sorted(recall)
fig,ax=plt.subplots(figsize=(5,4))
ax.bar([str(k) for k in ks],[recall[k] for k in ks]); ax.set_ylim(0,1)
ax.set_xlabel("k"); ax.set_ylabel("recall@k"); ax.set_title("Retrieval recall@k")
for i,k in enumerate(ks): ax.text(i,recall[k],f"{recall[k]:.2f}",ha="center",va="bottom")
fig.tight_layout(); fig.savefig("results_export/recall_at_k.png",dpi=150); plt.show()

# confusion
mat=np.array([[cm[a][b] for b in labels] for a in labels],float)
norm=mat/np.clip(mat.sum(1,keepdims=True),1,None)
fig,ax=plt.subplots(figsize=(4.5,4))
im=ax.imshow(norm,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel("predicted"); ax.set_ylabel("gold"); ax.set_title("Answer decision confusion")
for i in range(3):
    for j in range(3):
        ax.text(j,i,f"{int(mat[i,j])}",ha="center",va="center",
                color="white" if norm[i,j]>0.5 else "black")
fig.colorbar(im); fig.tight_layout(); fig.savefig("results_export/answer_confusion.png",dpi=150); plt.show()

with open("results_export/metrics.json","w") as f:
    json.dump({"recall_at_k":recall,"answer_accuracy":acc,"n_test":len(test),
               "generator":BACKEND},f,indent=2)
with open("results_export/predictions.jsonl","w") as f:
    for p in preds: f.write(json.dumps(p,ensure_ascii=False)+"\n")
print("saved.")


In [ ]:
import shutil
shutil.make_archive("results_export","zip","results_export")
from google.colab import files
files.download("results_export.zip")


---
받은 zip의 그림을 repo `results/`에, `metrics.json`의 recall@5와
answer accuracy를 README/포트폴리오 표에 기입하면 끝.
